# 🌫️ AQI 3-Day Forecast Pipeline
### Feature Engineering · SMOTE · XGBoost · SHAP Analysis

**Dataset:** Hourly AQI readings (Feb–May 2025)  
**Goal:** Predict AQI category for the next 72 hours  
**Model:** XGBoost Classifier with SMOTE oversampling + SHAP-driven feature selection


In [ ]:
# Install required libraries (run once)
import subprocess, sys
pkgs = ["xgboost", "shap", "imbalanced-learn", "lightgbm"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=False)
print("✅ All packages ready")


## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, mean_absolute_error
)
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

# ── Config ──────────────────────────────────────────────────
DATA_PATH        = "/mnt/user-data/uploads/aqi_data.csv"   # ← change if needed
POLLUTANTS       = ["co", "no", "no2", "o3", "so2", "pm2_5", "pm10", "nh3"]
TARGET           = "aqi"
FORECAST_HORIZON = 72          # 3 days × 24 hours
RANDOM_STATE     = 42
LAG_HOURS        = [1, 3, 6, 12, 24]
ROLL_WINDOWS     = [6, 12, 24]
TOP_N_FEATURES   = 40

AQI_LABELS  = {2: "Good", 3: "Moderate", 4: "Unhealthy (Sensitive)", 5: "Unhealthy"}
AQI_COLORS  = {2: "#00e400", 3: "#ffff00", 4: "#ff7e00", 5: "#ff0000"}

print("✅ Imports done")


## 2. Load & Clean Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
df = df.drop(columns=["_id", "timestamp"], errors="ignore")

# Clip outliers (3×IQR per pollutant)
for col in POLLUTANTS:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    fence = 3 * (Q3 - Q1)
    df[col] = df[col].clip(lower=Q1 - fence, upper=Q3 + fence)

print(f"Shape: {df.shape}")
print(f"Date range: {df['datetime'].min().date()} → {df['datetime'].max().date()}")
print("\nAQI class distribution:")
print(df[TARGET].value_counts().sort_index().rename(AQI_LABELS))
df.head()


## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 10), facecolor="#0d1117")
axes = axes.flatten()

for i, col in enumerate(POLLUTANTS + [TARGET]):
    ax = axes[i]
    ax.set_facecolor("#161b22")
    for sp in ax.spines.values():
        sp.set_edgecolor("#30363d")
    if col == TARGET:
        counts = df[col].value_counts().sort_index()
        colors = [AQI_COLORS[k] for k in counts.index]
        ax.bar([AQI_LABELS[k] for k in counts.index], counts.values, color=colors, alpha=0.85)
        ax.set_title("AQI Class Distribution", color="#e6edf3", fontsize=10, fontweight="bold")
        ax.tick_params(colors="#e6edf3", labelsize=7)
        ax.set_xticklabels([AQI_LABELS[k] for k in counts.index], rotation=20, ha="right")
    else:
        ax.plot(df["datetime"], df[col], color="#58a6ff", linewidth=0.5, alpha=0.8)
        ax.set_title(col.upper(), color="#e6edf3", fontsize=10, fontweight="bold")
        ax.tick_params(colors="#e6edf3", labelsize=7)
    ax.grid(color="#30363d", linestyle="--", linewidth=0.4, alpha=0.5)
    ax.yaxis.label.set_color("#e6edf3")

plt.suptitle("Pollutant Time Series + AQI Class Balance", color="#e6edf3",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 4. Feature Engineering

We create **131 features** across 5 categories:

| Category | Features |
|---|---|
| Cyclical time | hour/dow/month as sin+cos pairs |
| Context flags | rush-hour, night, weekend |
| Lag features | 1h, 3h, 6h, 12h, 24h for all pollutants + AQI |
| Rolling stats | mean + std over 6h, 12h, 24h windows |
| Domain interactions | pm_ratio, oxidants, combustion, cross-pollutant products |


In [ ]:
dt = df["datetime"]

# Temporal
df["hour"]        = dt.dt.hour
df["day_of_week"] = dt.dt.dayofweek
df["month"]       = dt.dt.month
df["day_of_year"] = dt.dt.dayofyear
df["week"]        = dt.dt.isocalendar().week.astype(int)

# Cyclical encoding
df["hour_sin"]   = np.sin(2 * np.pi * df["hour"]        / 24)
df["hour_cos"]   = np.cos(2 * np.pi * df["hour"]        / 24)
df["dow_sin"]    = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"]    = np.cos(2 * np.pi * df["day_of_week"] / 7)
df["month_sin"]  = np.sin(2 * np.pi * df["month"]       / 12)
df["month_cos"]  = np.cos(2 * np.pi * df["month"]       / 12)

# Context flags
df["is_rush_morning"] = df["hour"].between(7, 10).astype(int)
df["is_rush_evening"] = df["hour"].between(17, 20).astype(int)
df["is_night"]        = ((df["hour"] >= 22) | (df["hour"] <= 4)).astype(int)
df["is_weekend"]      = (df["day_of_week"] >= 5).astype(int)

# Lag features
for col in POLLUTANTS + [TARGET]:
    for lag in LAG_HOURS:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)

# Rolling statistics
for col in POLLUTANTS:
    for w in ROLL_WINDOWS:
        df[f"{col}_roll{w}_mean"] = df[col].shift(1).rolling(w).mean()
        df[f"{col}_roll{w}_std"]  = df[col].shift(1).rolling(w).std()
    df[f"{col}_ewm12"] = df[col].shift(1).ewm(span=12, adjust=False).mean()

# Domain interaction features
df["pm_ratio"]     = df["pm2_5"] / (df["pm10"] + 1e-6)
df["oxidants"]     = df["no2"] + df["o3"]
df["combustion"]   = df["co"] + df["no"]
df["pm2_5_x_no2"] = df["pm2_5"] * df["no2"]
df["so2_x_pm10"]  = df["so2"]   * df["pm10"]

# AQI trend
df["aqi_trend3"] = df[TARGET].diff(3)
df["aqi_trend6"] = df[TARGET].diff(6)

# Drop NaN rows from lag creation
df = df.dropna().reset_index(drop=True)

feature_cols = [c for c in df.columns
                if c not in ["datetime", TARGET] and not c.startswith("_")]
print(f"✅ Total features: {len(feature_cols)}")
print(f"   Rows after lag drop: {len(df)}")
print(f"\nSample features: {feature_cols[:10]} ...")


## 5. Time-Aware Train / Test Split

In [ ]:
SPLIT_IDX = len(df) - FORECAST_HORIZON
train_df  = df.iloc[:SPLIT_IDX]
test_df   = df.iloc[SPLIT_IDX:]

X_train, y_train = train_df[feature_cols].values, train_df[TARGET].values
X_test,  y_test  = test_df[feature_cols].values,  test_df[TARGET].values

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print(f"Train: {len(X_train)} rows  |  Test (72h hold-out): {len(X_test)} rows")
print(f"Train class dist: {dict(zip(le.classes_, np.bincount(y_train_enc)))}")
print(f"Test  class dist: {dict(zip(*np.unique(y_test_enc, return_counts=True)))}")


## 6. Class Imbalance — SMOTE

The dataset has a heavy skew toward AQI=3 (Moderate, 881 samples) vs AQI=2 (Good, 150 samples).  
SMOTE synthesizes new minority-class samples in feature space to balance training.


In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train_enc)

print(f"Before SMOTE: {len(X_train)} samples")
print(f"After  SMOTE: {len(X_resampled)} samples")
print(f"Resampled dist: {dict(zip(le.classes_, np.bincount(y_resampled)))}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="#0d1117")
for ax, counts, title in zip(
    axes,
    [np.bincount(y_train_enc), np.bincount(y_resampled)],
    ["Before SMOTE", "After SMOTE"]
):
    ax.set_facecolor("#161b22")
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")
    colors = [AQI_COLORS[c] for c in le.classes_]
    ax.bar([AQI_LABELS[c] for c in le.classes_], counts, color=colors, alpha=0.85)
    ax.set_title(title, color="#e6edf3", fontweight="bold")
    ax.tick_params(colors="#e6edf3")
    ax.grid(color="#30363d", linestyle="--", alpha=0.4)
    ax.set_xticklabels([AQI_LABELS[c] for c in le.classes_], rotation=15, ha="right", color="#e6edf3")
plt.suptitle("Class Balance Before vs After SMOTE", color="#e6edf3", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. XGBoost Training

Why XGBoost?
- Handles mixed feature types natively
- Built-in regularization (L1 + L2) prevents overfitting
- `hist` tree method is fast on tabular data
- Early stopping avoids over-training


In [ ]:
params = dict(
    n_estimators      = 800,
    max_depth         = 7,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    min_child_weight  = 3,
    gamma             = 0.1,
    objective         = "multi:softmax",
    num_class         = len(le.classes_),
    eval_metric       = "merror",
    use_label_encoder = False,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    tree_method       = "hist",
    early_stopping_rounds = 50,
)

model = xgb.XGBClassifier(**params)
model.fit(
    X_resampled, y_resampled,
    eval_set=[(X_test, y_test_enc)],
    verbose=50,
)
print(f"\n✅ Best iteration: {model.best_iteration}")


## 8. Evaluation on 72-Hour Hold-Out

In [ ]:
y_pred_enc = model.predict(X_test)
y_pred     = le.inverse_transform(y_pred_enc)
y_true     = y_test

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average="weighted")
mae = mean_absolute_error(y_true, y_pred)

print(f"Accuracy       : {acc:.4f}")
print(f"Weighted F1    : {f1:.4f}")
print(f"Mean Abs Error : {mae:.4f} AQI classes")

present_classes = sorted(set(y_true) | set(y_pred))
print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    labels=present_classes,
    target_names=[AQI_LABELS[c] for c in present_classes]
))

# Cross-validation
tscv = TimeSeriesSplit(n_splits=5)
cv_params = {k: v for k, v in params.items() if k != "early_stopping_rounds"}
cv_scores = cross_val_score(
    xgb.XGBClassifier(**cv_params), X_train, y_train_enc,
    cv=tscv, scoring="accuracy", n_jobs=-1
)
print(f"Time-Series CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), facecolor="#0d1117")
ax.set_facecolor("#161b22")
for sp in ax.spines.values(): sp.set_edgecolor("#30363d")

cm_classes = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=cm_classes)
im = ax.imshow(cm, cmap="YlOrRd")
plt.colorbar(im, ax=ax, fraction=0.04)
ticks = [AQI_LABELS[c] for c in cm_classes]
ax.set_xticks(range(len(cm_classes))); ax.set_xticklabels(ticks, rotation=25, ha="right", color="#e6edf3")
ax.set_yticks(range(len(cm_classes))); ax.set_yticklabels(ticks, color="#e6edf3")
ax.set_xlabel("Predicted", color="#e6edf3"); ax.set_ylabel("Actual", color="#e6edf3")
ax.set_title("Confusion Matrix — 72h Hold-Out", color="#e6edf3", fontweight="bold")
for i in range(len(cm_classes)):
    for j in range(len(cm_classes)):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13, fontweight="bold",
                color="white" if cm[i, j] > cm.max() * 0.5 else "black")
plt.tight_layout()
plt.show()


## 9. SHAP Feature Importance Analysis

SHAP (SHapley Additive exPlanations) explains **why** the model made each prediction  
by attributing a contribution score to every feature per sample.


In [ ]:
explainer  = shap.TreeExplainer(model)
shap_data  = np.vstack([X_train[-200:], X_test])
shap_vals  = explainer.shap_values(shap_data)

# Multi-class: average |SHAP| across all classes
if isinstance(shap_vals, list):
    mean_abs_shap = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
elif shap_vals.ndim == 3:
    mean_abs_shap = np.abs(shap_vals).mean(axis=2)
else:
    mean_abs_shap = np.abs(shap_vals)

shap_importance = pd.DataFrame({
    "feature":       feature_cols,
    "mean_abs_shap": mean_abs_shap.mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("Top 25 SHAP Features:")
print(shap_importance.head(25).to_string(index=False))


In [ ]:
# SHAP bar plot — Top 25
top25 = shap_importance.head(25)
fig, ax = plt.subplots(figsize=(10, 8), facecolor="#0d1117")
ax.set_facecolor("#161b22")
for sp in ax.spines.values(): sp.set_edgecolor("#30363d")

bar_colors = ["#58a6ff" if i < 5 else "#3fb950" if i < 12 else "#8b949e" for i in range(25)]
ax.barh(range(25), top25["mean_abs_shap"].values[::-1], color=bar_colors[::-1], height=0.7)
ax.set_yticks(range(25))
ax.set_yticklabels(top25["feature"].values[::-1], color="#e6edf3", fontsize=9)
ax.set_xlabel("Mean |SHAP Value|", color="#e6edf3")
ax.set_title("Top 25 Features by SHAP Importance", color="#e6edf3", fontsize=12, fontweight="bold")
ax.tick_params(colors="#e6edf3")
ax.grid(color="#30363d", linestyle="--", alpha=0.4, axis="x")
plt.tight_layout()
plt.show()


## 10. SHAP-Driven Feature Selection + Retraining

Retrain using only the top 40 SHAP-selected features.  
Fewer features = faster inference + less overfitting risk.


In [ ]:
top_features = shap_importance.head(TOP_N_FEATURES)["feature"].tolist()
feat_idx     = [feature_cols.index(f) for f in top_features]

X_tr_top = X_resampled[:, feat_idx]
X_te_top = X_test[:, feat_idx]

model_shap = xgb.XGBClassifier(**{k: v for k, v in params.items()
                                   if k != "early_stopping_rounds"})
model_shap.set_params(n_estimators=model.best_iteration + 1)
model_shap.fit(X_tr_top, y_resampled)

y_pred_shap     = model_shap.predict(X_te_top)
y_pred_shap_dec = le.inverse_transform(y_pred_shap)

acc2 = accuracy_score(y_true, y_pred_shap_dec)
f1_2 = f1_score(y_true, y_pred_shap_dec, average="weighted")
mae2 = mean_absolute_error(y_true, y_pred_shap_dec)
print(f"SHAP-Pruned Model  →  Accuracy: {acc2:.4f}  |  F1: {f1_2:.4f}  |  MAE: {mae2:.4f}")

# Pick final model
if f1_2 >= f1 - 0.005:
    final_model, final_feats, final_fidx = model_shap, top_features, feat_idx
    print("→ Using SHAP-pruned model (fewer features, same performance)")
else:
    final_model, final_feats, final_fidx = model, feature_cols, list(range(len(feature_cols)))
    print("→ Using full-feature model")


## 11. 3-Day Recursive Forecast (72 Hours)

We predict hour-by-hour, feeding each prediction back as a lag feature for the next step.


In [ ]:
forecast_rows = []
df_ext        = df.copy()
last_time     = df_ext["datetime"].iloc[-1]

for h in range(1, FORECAST_HORIZON + 1):
    next_time = last_time + pd.Timedelta(hours=h)
    row = {}

    # Temporal
    row["hour"]        = next_time.hour
    row["day_of_week"] = next_time.dayofweek
    row["month"]       = next_time.month
    row["day_of_year"] = next_time.dayofyear
    row["week"]        = next_time.isocalendar()[1]
    row["hour_sin"]    = np.sin(2 * np.pi * row["hour"]        / 24)
    row["hour_cos"]    = np.cos(2 * np.pi * row["hour"]        / 24)
    row["dow_sin"]     = np.sin(2 * np.pi * row["day_of_week"] / 7)
    row["dow_cos"]     = np.cos(2 * np.pi * row["day_of_week"] / 7)
    row["month_sin"]   = np.sin(2 * np.pi * row["month"]       / 12)
    row["month_cos"]   = np.cos(2 * np.pi * row["month"]       / 12)
    row["is_rush_morning"] = int(7  <= row["hour"] <= 10)
    row["is_rush_evening"] = int(17 <= row["hour"] <= 20)
    row["is_night"]        = int(row["hour"] >= 22 or row["hour"] <= 4)
    row["is_weekend"]      = int(row["day_of_week"] >= 5)

    # Lags
    for col in POLLUTANTS + [TARGET]:
        for lag in LAG_HOURS:
            idx = -lag
            row[f"{col}_lag{lag}"] = df_ext[col].iloc[idx] if abs(idx) <= len(df_ext) else df_ext[col].iloc[-1]

    # Rolling
    for col in POLLUTANTS:
        for w in ROLL_WINDOWS:
            s = df_ext[col].iloc[-w:] if len(df_ext) >= w else df_ext[col]
            row[f"{col}_roll{w}_mean"] = s.mean()
            row[f"{col}_roll{w}_std"]  = s.std() if len(s) > 1 else 0.0
        row[f"{col}_ewm12"] = df_ext[col].ewm(span=12, adjust=False).mean().iloc[-1]

    # Interactions
    row["pm_ratio"]     = df_ext["pm2_5"].iloc[-1] / (df_ext["pm10"].iloc[-1] + 1e-6)
    row["oxidants"]     = df_ext["no2"].iloc[-1]  + df_ext["o3"].iloc[-1]
    row["combustion"]   = df_ext["co"].iloc[-1]   + df_ext["no"].iloc[-1]
    row["pm2_5_x_no2"] = df_ext["pm2_5"].iloc[-1] * df_ext["no2"].iloc[-1]
    row["so2_x_pm10"]  = df_ext["so2"].iloc[-1]   * df_ext["pm10"].iloc[-1]
    row["aqi_trend3"]  = df_ext[TARGET].iloc[-1] - df_ext[TARGET].iloc[-4] if len(df_ext) >= 4 else 0
    row["aqi_trend6"]  = df_ext[TARGET].iloc[-1] - df_ext[TARGET].iloc[-7] if len(df_ext) >= 7 else 0

    x_vec = np.array([row.get(f, 0.0) for f in feature_cols]).reshape(1, -1)
    x_sel = x_vec[:, final_fidx]

    pred_enc   = final_model.predict(x_sel)[0]
    pred_proba = final_model.predict_proba(x_sel)[0]
    pred_aqi   = int(le.inverse_transform([pred_enc])[0])
    confidence = float(pred_proba.max())

    forecast_rows.append({
        "datetime":      next_time,
        "predicted_aqi": pred_aqi,
        "label":         AQI_LABELS[pred_aqi],
        "confidence_%":  round(confidence * 100, 1),
        **{f"p(AQI={le.classes_[i]})": round(float(p), 3) for i, p in enumerate(pred_proba)}
    })

    new_row = {c: np.nan for c in df_ext.columns}
    new_row["datetime"] = next_time
    new_row[TARGET]     = pred_aqi
    for col in POLLUTANTS:
        new_row[col] = df_ext[col].iloc[-1]
    df_ext = pd.concat([df_ext, pd.DataFrame([new_row])], ignore_index=True)

forecast_df = pd.DataFrame(forecast_rows)
print("✅ Forecast complete")
forecast_df.iloc[::6][["datetime","predicted_aqi","label","confidence_%"]]


## 12. Forecast Visualization

In [ ]:
DARK_BG = "#0d1117"; CARD_BG = "#161b22"; GRID_CLR = "#30363d"; TEXT_CLR = "#e6edf3"
ACC_CLR = "#58a6ff"; WARN_CLR = "#f0883e"

fig = plt.figure(figsize=(20, 18), facecolor=DARK_BG)
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3,
                        top=0.93, bottom=0.06, left=0.07, right=0.97)

def style_ax(ax, title=""):
    ax.set_facecolor(CARD_BG)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_CLR)
    ax.tick_params(colors=TEXT_CLR, labelsize=8)
    ax.xaxis.label.set_color(TEXT_CLR); ax.yaxis.label.set_color(TEXT_CLR)
    ax.grid(color=GRID_CLR, linestyle="--", linewidth=0.5, alpha=0.6)
    if title: ax.set_title(title, color=TEXT_CLR, fontsize=11, fontweight="bold", pad=8)

fig.text(0.5, 0.97, "AQI 3-Day Forecast Dashboard",
         ha="center", color=TEXT_CLR, fontsize=16, fontweight="bold")
fig.text(0.5, 0.953, f"XGBoost + SMOTE  |  Accuracy: {acc2:.2%}  |  F1: {f1_2:.4f}  |  MAE: {mae2:.4f}",
         ha="center", color="#8b949e", fontsize=10)

# Plot 1: Historical + Forecast
ax1 = fig.add_subplot(gs[0, :])
style_ax(ax1, "Historical AQI (last 30 days) + 72h Forecast")
hist30 = df.tail(30 * 24)
ax1.bar(range(len(hist30)), hist30[TARGET], color=[AQI_COLORS[v] for v in hist30[TARGET]],
        width=1, alpha=0.7, label="Historical")
fc_x = range(len(hist30), len(hist30) + len(forecast_df))
ax1.bar(fc_x, forecast_df["predicted_aqi"], color=[AQI_COLORS[v] for v in forecast_df["predicted_aqi"]],
        width=1, alpha=0.9, label="Forecast (72h)")
ax1.axvline(len(hist30), color=WARN_CLR, linestyle="--", linewidth=1.5, label="Forecast start")
ax1.set_yticks([2,3,4,5]); ax1.set_yticklabels(["Good","Moderate","Unhlthy(S)","Unhealthy"], color=TEXT_CLR, fontsize=8)
ax1.legend(facecolor=CARD_BG, edgecolor=GRID_CLR, labelcolor=TEXT_CLR)

# Plot 2: 72h forecast with confidence band
ax2 = fig.add_subplot(gs[1, :])
style_ax(ax2, "72-Hour Forecast with Confidence Band")
x_fc = np.arange(len(forecast_df))
y_fc = forecast_df["predicted_aqi"].values
conf = forecast_df["confidence_%"].values / 100
ax2.bar(x_fc, y_fc, color=[AQI_COLORS[v] for v in forecast_df["predicted_aqi"]], width=0.9, alpha=0.8)
ax2.plot(x_fc, y_fc, color=ACC_CLR, linewidth=1.5)
ax2.fill_between(x_fc, y_fc-(1-conf), y_fc+(1-conf), alpha=0.15, color=ACC_CLR, label="Confidence band")
for day in [24, 48]:
    ax2.axvline(day, color=GRID_CLR, linewidth=1)
    ax2.text(day+0.5, 5.2, f"Day {day//24+1}", color="#8b949e", fontsize=8)
xt = list(range(0, 72, 6))
ax2.set_xticks(xt)
ax2.set_xticklabels([forecast_df["datetime"].iloc[i].strftime("%m/%d %Hh") for i in xt],
                     rotation=45, ha="right", color=TEXT_CLR, fontsize=7)
ax2.set_yticks([2,3,4,5]); ax2.set_yticklabels(["Good","Moderate","Unhlthy(S)","Unhealthy"], color=TEXT_CLR, fontsize=8)
ax2.set_ylim(1.5, 5.8); ax2.legend(facecolor=CARD_BG, edgecolor=GRID_CLR, labelcolor=TEXT_CLR)

# Plot 3: SHAP Top 20
ax3 = fig.add_subplot(gs[2, 0])
style_ax(ax3, "Top 20 SHAP Feature Importance")
top20 = shap_importance.head(20)
colors_bar = ["#58a6ff" if i < 5 else "#3fb950" if i < 10 else "#8b949e" for i in range(20)]
ax3.barh(range(20), top20["mean_abs_shap"].values[::-1], color=colors_bar[::-1], height=0.7)
ax3.set_yticks(range(20)); ax3.set_yticklabels(top20["feature"].values[::-1], color=TEXT_CLR, fontsize=8)
ax3.set_xlabel("Mean |SHAP|", color=TEXT_CLR)

# Plot 4: Probability heatmap
ax4 = fig.add_subplot(gs[2, 1])
style_ax(ax4, "Forecast Probability Heatmap")
prob_cols = [c for c in forecast_df.columns if c.startswith("p(AQI=")]
prob_mat  = forecast_df[prob_cols].values.T
im = ax4.imshow(prob_mat, aspect="auto", cmap="hot", vmin=0, vmax=1)
plt.colorbar(im, ax=ax4, fraction=0.04)
ax4.set_yticks(range(len(prob_cols)))
ax4.set_yticklabels([c.replace("p(","").replace(")","") for c in prob_cols], color=TEXT_CLR, fontsize=9)
ax4.set_xticks(range(0,72,12)); ax4.set_xticklabels([f"h{i}" for i in range(0,72,12)], color=TEXT_CLR, fontsize=8)
ax4.set_xlabel("Forecast Hour", color=TEXT_CLR)

plt.show()


## 13. Save Outputs

In [ ]:
import os, joblib

OUT = "/mnt/user-data/outputs"
os.makedirs(OUT, exist_ok=True)

forecast_df.to_csv(f"{OUT}/aqi_3day_forecast.csv", index=False)
shap_importance.to_csv(f"{OUT}/shap_feature_importance.csv", index=False)
final_model.save_model(f"{OUT}/aqi_xgb_model.json")

print("✅ Saved:")
print(f"   {OUT}/aqi_3day_forecast.csv")
print(f"   {OUT}/shap_feature_importance.csv")
print(f"   {OUT}/aqi_xgb_model.json")
print(f"\n📊 Final Results:")
print(f"   Features engineered : {len(feature_cols)}")
print(f"   SHAP-selected feats : {TOP_N_FEATURES}")
print(f"   Accuracy            : {acc2:.2%}")
print(f"   Weighted F1         : {f1_2:.4f}")
print(f"   MAE                 : {mae2:.4f} AQI classes")
print(f"   Forecast horizon    : 72 hours (3 days)")
